# 웹 문서 요약 with RAG — Map-Reduce 패턴으로 긴 문서 처리

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Map-Reduce** 패턴으로 LLM 컨텍스트 한계를 초과하는 문서 요약하기
- 커스텀 프롬프트로 Map 단계(청크 요약)와 Reduce 단계(통합 요약)를 각각 설정하기
- 요약본과 원본 청크를 함께 벡터스토어에 저장하는 하이브리드 RAG 구축하기

## 사전 지식

- 01-RAG-Web-Based.ipynb의 웹 문서 크롤링 방법
- LangChain의 `|` 연산자로 체인 연결하는 방법

---

```mermaid
flowchart TD
    WEB[웹 페이지]:::input --> C[청크 분할<br/>3000자]:::process
    subgraph MAP[Map 단계]
        C --> M1[청크1 요약]:::process
        C --> M2[청크2 요약]:::process
        C --> M3[청크3 요약]:::process
    end
    M1 --> R[Reduce 단계<br/>통합 요약]:::process
    M2 --> R
    M3 --> R
    R --> VS[(벡터스토어<br/>원본+요약)]:::storage
    Q[사용자 질문]:::input --> VS
    VS --> A[정확한 답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## Map-Reduce 요약 전략

| 단계 | 역할 | 입력 | 출력 |
|------|------|------|------|
| **Map** | 각 청크를 독립적으로 요약 | 청크 하나 | 짧은 요약 |
| **Reduce** | 모든 요약을 하나로 통합 | 모든 Map 결과 | 최종 요약 |

> **실무 팁**: `chunk_size=3000`처럼 크게 설정하면 Map 단계 횟수가 줄어 API 비용이 절약돼요. 단, 청크가 너무 크면 LLM이 핵심을 놓칠 수 있으니 1000~3000자 범위가 적당해요.

## 왜 요약이 필요한가?

### 문제 상황

- 웹 페이지는 수만 자의 긴 텍스트 포함
- 모든 내용을 검색 대상으로 하면 노이즈 증가
- LLM 컨텍스트 윈도우 제한

### 해결책: 요약 기반 RAG

```
긴 웹 문서 (10,000자)
    ↓
자동 요약 (2,000자)
    ↓
벡터 검색 (효율적)
    ↓
정확한 답변
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser



# 요약을 위한 분할 (청크 크기를 크게)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,  # 요약용으로 큰 청크
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(split_docs)}")
if split_docs:
    print(f"첫 번째 청크 길이: {len(split_docs[0].page_content)} 문자")
else:
    print("⚠️ 충분한 텍스트를 가져오지 못해 청크가 생성되지 않았습니다.")


In [ ]:
# ---------------------------------------------------
# WebBaseLoader로 Wikipedia 딥러닝 페이지 크롤링
# ---------------------------------------------------
# ============================================================
# TODO: WebBaseLoader와 SoupStrainer로 Wikipedia 딥러닝 페이지를 로드하세요
# 힌트: bs_kwargs={"parse_only": bs4.SoupStrainer("div", attrs={"class": "mw-parser-output"})}
# 예상 결과: 문서 수, 총 문자 수 출력
# ============================================================

# Wikipedia 긴 문서 로드
# SoupStrainer: 본문 영역만 추출하여 네비게이션 등 HTML 노이즈 제거
loader = WebBaseLoader(
    web_paths=("https://ko.wikipedia.org/wiki/딥_러닝",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": "mw-parser-output"}
        )
    ),
)

docs = loader.load()

print(f"문서 수: {len(docs)}")
print(f"문서 길이: {len(docs[0].page_content):,} 문자")
print(f"\n문서 미리보기:\n{docs[0].page_content[:300]}...")

## 2. 문서 분할 (요약 준비)

In [ ]:
# ---------------------------------------------------
# 문서를 Map-Reduce 요약에 적합한 크기로 분할
# ---------------------------------------------------
# ============================================================
# TODO: chunk_size=3000으로 문서를 분할하세요
# 힌트: RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)
# 예상 결과: 분할된 청크 수와 첫 청크 길이 출력
# ============================================================

# 요약용 청크 — chunk_size를 크게 설정해 Map 단계 횟수와 API 비용 절약
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,  # 요약용으로 큰 청크 (세부 검색용보다 크게)
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(split_docs)}")
if split_docs:
    print(f"첫 번째 청크 길이: {len(split_docs[0].page_content)} 문자")
else:
    print("⚠️ 충분한 텍스트를 가져오지 못해 청크가 생성되지 않았습니다.")

## 3. Map-Reduce 요약

### Map-Reduce 패턴

```
청크 1  →  요약 1  \
청크 2  →  요약 2   → 최종 통합 요약
청크 3  →  요약 3  /
```

- **Map**: 각 청크를 독립적으로 요약
- **Reduce**: 모든 요약을 하나로 통합

In [ ]:
# ---------------------------------------------------
# Map-Reduce 패턴으로 긴 문서 요약
# ---------------------------------------------------
# ============================================================
# TODO: map_chain으로 각 청크를 요약(Map)하고, reduce_chain으로 통합(Reduce)하세요
# 힌트: map_summaries = [map_chain.invoke({"text": doc.page_content}) for doc in split_docs]
# 예상 결과: 전체 문서 요약 출력 및 압축률 표시
# ============================================================


## 4. 맞춤 요약 프롬프트

기본 요약 대신, 특정 관점으로 요약할 수 있습니다.

In [ ]:
# ---------------------------------------------------
# 커스텀 Map/Reduce 프롬프트로 핵심 내용 위주 요약
# ---------------------------------------------------
# ============================================================
# TODO: 커스텀 map_prompt와 reduce_prompt를 정의하고 요약 파이프라인을 실행하세요
# 힌트: custom_map_chain = map_prompt | llm | StrOutputParser()
# 예상 결과: 커스텀 요약 결과 출력
# ============================================================


## 5. 요약 기반 RAG 구축

요약된 내용을 벡터스토어에 저장하여 효율적인 검색을 수행합니다.

In [ ]:
# ---------------------------------------------------
# 원본 청크 + 요약 문서를 통합한 하이브리드 RAG 구축
# ---------------------------------------------------
# ============================================================
# TODO: 원본 split_docs와 요약 summary_doc을 합쳐 FAISS 벡터스토어와 RAG 체인을 만드세요
# 힌트: all_docs = split_docs + [summary_doc], FAISS.from_documents(all_docs, embeddings)
# 예상 결과: "요약 기반 RAG 시스템 구축 완료" 출력
# ============================================================


## 6. 질의응답 테스트

In [ ]:
# ---------------------------------------------------
# 요약 기반 RAG 실행 — 질문 1
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain.invoke()로 딥러닝에 대해 질문하세요
# 힌트: rag_chain.invoke("딥러닝이란 무엇인가요?")
# 예상 결과: 요약 또는 원본 문서 기반 답변 출력
# ============================================================


In [ ]:
# ---------------------------------------------------
# 요약 기반 RAG 실행 — 질문 2
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain.invoke()로 주요 활용 사례를 질문하세요
# 힌트: rag_chain.invoke("요약 내용을 기반으로 주요 활용 사례를 알려주세요.")
# 예상 결과: 요약 문서에서 검색된 정보 기반 답변 출력
# ============================================================


## 💡 핵심 정리

### Map-Reduce 요약의 장점

1. **긴 문서 처리**: LLM 컨텍스트 제한 극복
2. **병렬 처리**: 각 청크를 독립적으로 요약 (속도 향상 가능)
3. **유연성**: Map/Reduce 프롬프트 커스터마이징

### 요약 체인 타입

| 타입 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **stuff** | 모든 문서를 한 번에 | 빠름 | 긴 문서 불가 |
| **map_reduce** | 청크별 요약 후 통합 | 긴 문서 가능 | 느림 |
| **refine** | 순차적 정제 요약 | 일관성 높음 | 매우 느림 |

### 실전 활용

- 뉴스 기사 요약 및 QA
- 논문/보고서 분석
- 웹 콘텐츠 요약 봇
- 긴 회의록 요약

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 패턴 | Map(청크별 독립 요약) → Reduce(전체 통합 요약) |
| 저장 전략 | 원본 청크 + 요약 청크 모두 VectorStore에 통합 |
| Map 단계 | 각 청크를 독립적으로 요약 — 병렬 처리 가능 |
| Reduce 단계 | Map 결과를 합쳐 최종 요약문 생성 |
| 장점 | LLM 컨텍스트 제한을 넘어서는 긴 문서 처리 가능 |

### Map-Reduce vs 단순 요약

| 방식 | 처리 가능 길이 | 일관성 | LLM 호출 수 |
|------|----------------|--------|-------------|
| 단순 요약 | 컨텍스트 내 | 높음 | 1회 |
| Map-Reduce | 무제한 | 중간 | 청크 수 + 1 |
| Refine (점진) | 무제한 | 높음 | 청크 수 × 2 |

### 6-7 RAG Process 시리즈 완성

| 노트북 | 핵심 기술 | 적합한 상황 |
|--------|-----------|-------------|
| 00-RAG-Basic | 8단계 기본 파이프라인 | RAG 처음 시작할 때 |
| 01-Web-Based | WebBaseLoader + BeautifulSoup | 웹 문서 실시간 QA |
| 02-Advanced | Ensemble + MMR | 검색 다양성이 필요할 때 |
| 03-Conversation | MessageHistory + 세션 | 멀티턴 대화 시스템 |
| 04-RAPTOR | 계층적 요약 인덱스 | 매우 긴 문서 전체 이해 |
| **05-Web-Summarization** | **Map-Reduce 요약** | **웹 문서 대규모 요약 + QA** |

6-5 Retriever부터 6-7 RAG Process까지 모든 핵심 RAG 기법을 학습했어요. 이제 실제 프로젝트에 다양한 기법을 조합해 적용해 보세요!